## Upload data to local Postgre SQL database

Cálculo de la ampacidad IEEE y CIGRE con la implementación UC. El script está diseñado para trabajar con los ficheros .csv descargados [actualmente se descargan en la carpeta e:\GE] de la web on-premise de GE [pestaña dedicada a REPORTS] agrupados por línea y definidos con fechas del 1 [00:00] al último día del mes [23:59], que deben tener un nombre con la estructura siguiente:

GE_XXXXMMDD_XXXXNNEE.csv   
XXXXX .- Año
MMDD .- Desde el mes MM y día DD

hasta el

XXXXX .- Año
NNEE .- Hasta el mes NN y día EE


Las columnas del fichero serán:

LineName;NodeName;TimeStamp;PhaseCurrent;PhasePhase;ConductorTemp;AmbientTemp;WindSpeed;WindDomDirection;WindAvgDirection;SolarRadiation;DewPoint;ServiceName;Clearance;IMAX;LoadMVA;MaxCapacityMVA



**author**: GTEA-UC  
**email**: mananam@unican.es  
**last update**: 26/12/2023

In [1]:
import sys 
print( sys.version)

3.9.15 (main, Nov 24 2022, 14:39:17) [MSC v.1916 64 bit (AMD64)]


In [2]:
import psycopg2
import pyodbc
import configparser # Config file
import pandas as pd
import datetime
from datetime import datetime, timedelta
import calendar
import time
from dateutil.relativedelta import relativedelta
from sqlalchemy import create_engine
import os

**Paquetes específicos DLR**

In [3]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from pvsystems import pvsystems
import matplotlib.pyplot as plt 


# Needed only during the development phase.
from importlib import reload
reload( cable)
reload( case)
reload( ieee738)
reload( cigre601)
reload( pvsystems)

<module 'pvsystems.pvsystems' from 'e:\\mario\\python\\pypacity\\pvsystems\\pvsystems.py'>

Upload the dataframe to the database table

Information for the connection to the database

In [4]:
# PostgreSQL connection parameters
db_params = {
    'host': 'localhost', # the database is in the same machine where this script is running
    'database': 'Iberdrola', # name of the database
    'user': 'postgres', # user
    'password': 'mananam05' # password
}

Create a connection to the database. 

Define the name of the table where the transactions will take place.

In [5]:
# Establish a connection to the PostgreSQL database using SQLAlchemy
connection_string = f"postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params['host']}/{db_params['database']}"
engine = create_engine(connection_string)

# Define the name of the table you want to create or replace
table_name = 'ucgeproc'

patha .- is the path to the folder where the information about the lines is storaged.

LineName .- is the array with the names of the lines. This array is defined according patha


In [6]:
patha = [ 'E:/GE_Iberdrola/']
#LineNamea = [ 'El Palmar-Espinardo']

file_case = "E:/mario/trabajos2/iberdrola_DTR/datos/case.xlsx" # Information about the cases
file_cable = "E:/mario/trabajos2/iberdrola_DTR/datos/cable.xlsx" # Information about the cables


case_dtypes={ 'LineName': str,
             'NodeName': str}


cable_dtypes={ 'LineName': str}


dfCases = pd.read_excel( file_case, header=0, dtype=case_dtypes)
dfCases['LineName'] = dfCases['LineName'].astype( str)
dfCases['NodeName'] = dfCases['NodeName'].astype( str)
dfCables = pd.read_excel( file_cable, header=0, dtype=cable_dtypes)
dfCables['LineName'] = dfCables['LineName'].astype( str)


In [7]:
dfCases

,LineName,NodeName,ANG_DEG,CDR_ELEV,Z1_DEG,CDR_LAT_DEG,TCDR,ALBEDO
0,EL PALMAR - ESPINARDO,10160,0.00,50.17,0.00,38.0,50.0,0.2
1,EL PALMAR - ESPINARDO,10166,0.00,46.00,0.00,38.0,50.0,0.2
2,EL PALMAR - ESPINARDO,10174,0.00,45.86,0.00,38.0,50.0,0.2
3,HELLÍN - CALASPARRA,10021,0.00,336.00,0.00,38.0,60.0,0.2
4,HELLÍN - CALASPARRA,10031,0.00,459.00,0.00,38.0,60.0,0.2
5,HELLÍN - CALASPARRA,10042,0.00,358.00,0.00,38.0,60.0,0.2
6,HELLÍN - CALASPARRA,10055,0.00,438.00,0.00,38.0,60.0,0.2
7,HELLÍN - CALASPARRA,10068,0.00,479.00,0.00,38.0,60.0,0.2
8,HELLÍN - CALASPARRA,10085,0.00,579.00,0.00,38.0,60.0,0.2
9,COLLADO - BUÑOL,10006-10007,31.24,358.47,31.24,39.0,65.0,0.2


In [8]:
dfCables

,LineName,Cstring,D,d,TLO,THI,TCDRMAX,RLO,RHI,EMISS,...,HNH,HEATOUT,HEATCORE,TotalS,CSteel20,CAlum20,BetaSteel20,BetaAlum20,mSteel,mAlum
0,EL PALMAR - ESPINARDO,LA-280,0.0218,0.00340,20.0,75.0,50.0,0.000119,0.000146,0.5,...,2.0,NaN,NaN,241.7,481.0,897.0,0.0001,0.000318,0.3977,0.2641
1,HELLÍN - CALASPARRA,LA-180,0.0175,0.00250,20.0,75.0,65.0,0.000196,0.000241,0.8,...,2.0,NaN,NaN,147.3,481.0,897.0,0.0001,0.000318,0.3977,0.2641
2,COLLADO - BUÑOL,LA-180,0.0175,0.00250,20.0,75.0,65.0,0.000196,0.000241,0.8,...,2.0,NaN,NaN,147.3,481.0,897.0,0.0001,0.000318,0.3977,0.2641
3,LA NUCIA - CALPE,LA-280,0.0218,0.00344,20.0,75.0,50.0,0.000120,0.000147,0.6,...,2.0,NaN,NaN,241.7,481.0,897.0,0.0001,0.000318,0.3977,0.2641
4,LA PLANA - VILLARREAL SUR,LA-280,0.0218,0.00344,20.0,75.0,50.0,0.000120,0.000147,0.6,...,2.0,NaN,NaN,241.7,481.0,897.0,0.0001,0.000318,0.3977,0.2641
5,OLIVA - VERGEL,LA-280,0.0218,0.00344,20.0,75.0,50.0,0.000120,0.000147,0.6,...,2.0,NaN,NaN,241.7,481.0,897.0,0.0001,0.000318,0.3977,0.2641
6,ROCAMORA - CARRUS,LA-280,0.0218,0.00344,20.0,75.0,50.0,0.000120,0.000147,0.6,...,2.0,NaN,NaN,241.7,481.0,897.0,0.0001,0.000318,0.3977,0.2641


In [9]:
def get_df2( data_frame, dfCables, dfCases, table_name, engine):
    
    column_names = [
            'LineName',
            'NodeName',
            'TimeStamp',
            'PhaseCurrent',
            'PhasePhase' ,
            'ConductorTemp' ,
            'AmbientTemp' ,
            'WindSpeed' ,
            'WindDomDirection' ,
            'WindAvgDirection' ,
            'SolarRadiation' ,
            'DewPoint' ,
            'ServiceName' ,
            'Clearance' ,
            'IMAX' , 
            'LoadMVA' ,
            'MaxCapacityMVA' ,
            'IEEE738' ,
            'CIGRE601' ,
            'ucIEEE738' ,
            'ucCIGRE601' ]
    
    
    TotalErrors = 0  
    TotalRows = len( data_frame) #Tamaño del array a procesar
    TotalCount = 1 # Contador de 1 a TotalRows
    PrintEach = 1000 # Imprime cada [PrientEach] iteracciones
    PrintControl = 1 # Contador para activar una vez cada [PrintEach]
    df2 = pd.DataFrame( columns = column_names) # define a new dataframe with the columns defined by the array column_names.
    for index, row in data_frame.iterrows():
        
        LineName = str( row['LineName'])
        NodeName = str( row['NodeName'])
        #datetime_str = str(row['Date']) + "  " + str(row['Time'])
        #datetime_obj = datetime.strptime( datetime_str, "%d/%m/%Y %H:%M")
        #TimeStamp = datetime_obj.strftime('%Y-%m-%d %H:%M:%S') 
        TimeStamp = str( row['TimeStamp'])


        dfCable = dfCables[ dfCables['LineName']==LineName]
        #print("***************************************")
        #print(LineName)
        #print(dfCable)
        dfCase = dfCases[ dfCases['NodeName']==NodeName]
        #print("-----------------------------------------")
        #print('\r' + NodeName)
        #print(dfCase)
        #print("***************************************")
        #break

        AmbientTemp = float( row['AmbientTemp'])
        #CondTemperature = float(row['ConductorTemp'])
        WindspeedAVG = float(row['WindSpeed'])
        WindDirectionAVG = float(row['WindAvgDirection'])
        WindDirectionDominat = float(row['WindDomDirection'])
        SolarRadiation = float(row['SolarRadiation'])
        DewPointTemperature = float(row['DewPoint'])
        WindSonic = 0.0 #float(row['WindSonic Windspeed AVG'])
        WindSonicWindDirectionAVG = 0.0 #float(row['WindSonic Wind Direction AVG'])
        WindSonicWindDirectionDominant = 0.0 #float(row['WindSonic Wind Direction Dominant'])
        PhaseCurrent = float(row['PhaseCurrent'])
        ConductorTemp = float(row['ConductorTemp'])
        RCCIEEE = 0.0 #float(row['RCC IEEE'])
        RCCCIGRE = float(row['IMAX'])
        Clearance = 0.0 #float(row['CLEARANCE'])
        RIME = 0.0 #float(row['RIME'])
    

        NSELECT = 2 
        Cable1 = cable.Cable()
        #c_db, error = Cable1.load_cable_db()
        #Cable1.set_cable( NSELECT, conductor = dfCable.at[1,'Cstring'])

        Cable1.D = 1000*float( dfCable['D'])
        #Cable1.C = 10.4
        Cable1.d = 1000*float( dfCable['d']) 
        Cable1.TLO = float( dfCable['TLO']) 
        Cable1.THI = float( dfCable['THI']) 
        #Cable1.TCDRMAX = 60.0
        Cable1.RLO = float( dfCable['RLO']) 
        Cable1.RHI = float( dfCable['RHI']) 

        Cable1.EMISS = float( dfCable['EMISS'])
        Cable1.ABSORP = float( dfCable['ABSORP'])


        Cable1.HNH = int( dfCable['HNH']) 
        #Cable1.HEATOUT = 357.9
        #Cable1.HEATCORE = 132.1
        Cable1.TotalS = float( dfCable[ 'TotalS']) 
        Cable1.CSteel20 = float( dfCable['CSteel20']) 
        Cable1.CAlum20 = float( dfCable[ 'CAlum20']) 
        Cable1.BetaSteel20 = float( dfCable[ 'BetaSteel20']) 
        Cable1.BetaAlum20 = float( dfCable[ 'BetaAlum20']) 
        Cable1.mSteel = float( dfCable[ 'mSteel']) 
        Cable1.mAlum = float( dfCable[ 'mAlum']) 


        Case1 = case.Case()
        Case1.demo( NSELECT)
        # Ambient conditions
        Case1.TAMB = AmbientTemp
        try:
            Case1.CDR_LAT_DEG = float( dfCase['CDR_LAT_DEG'])
        except Exception as e:
            print(NodeName)
            break
        
        Case1.ALBEDO = float( dfCase['ALBEDO'])
        Case1.beta = 0
        Case1.CDR_ELEV = float( dfCase['CDR_ELEV'])
        Case1.TCDRPRELOAD = float( dfCase['TCDR'])
        #Case1.TCDRMAX = 150
        #Case1.TCDR = 100.0
        Case1.SolarRadiation = SolarRadiation
        if WindspeedAVG < 0.1:
            WindspeedAVG += 0.01
        Case1.VWIND = WindspeedAVG 
        Case1.WINDANG_DEG = abs(WindDirectionAVG - (float( dfCase['ANG_DEG'])))
        Case1.Z1_DEG = float( dfCase['Z1_DEG'])
        Case1.Ns = 1.0
        Case1.SUN_TIME = 99 # solar Radiation measured (IEEE738)
        Case1.SOLAR = 0 # Solar Radiation measured (CIGRE601)
        #dia = int(datetime_obj.strftime('%d'))
        #mes = int(datetime_obj.strftime('%m') )
        #Case1.NDAY = PV1.DayOfYear( dia, mes) # 10th June
        #print("NDAY: " + str(Case1.NDAY))

        # IEEE 738
        X1 = ieee738.IEEE738()
        X1.Debug = 0
        X1.set_cable( Cable1)
        X1.set_case( Case1)
        X1.Case1.SORM = 1
        X1.ieee_738_2013()      
        #X1.output()
        ucIEEE738 = round( float(X1.Case1.TR), 2)

        # CIGRE TB 601
        X2 = cigre601.CIGRE601()
        X2.Debug = 0
        X2.set_error( 0) # no error
        X2.set_cable( Cable1)
        X2.set_case( Case1)
        X2.cigre601()    
        #X2.output()
        ucCIGRE601 =round( float(X2.Case1.TR), 2)
    
    
        new_row_data = {
            'LineName' : LineName,
            'NodeName' : NodeName,
            'TimeStamp' : TimeStamp,
            'PhaseCurrent' : PhaseCurrent,
            'PhasePhase' : ' ',
            'ConductorTemp' : ConductorTemp,
            'AmbientTemp' : AmbientTemp,
            'WindSpeed' : WindspeedAVG,
            'WindDomDirection' : WindDirectionDominat,
            'WindAvgDirection' : WindDirectionAVG,
            'SolarRadiation' : SolarRadiation,
            'DewPoint' : DewPointTemperature,
            'ServiceName' : ' ',
            'Clearance' : Clearance,
            'IMAX' : ' ',
            'LoadMVA' : ' ',
            'MaxCapacityMVA' : ' ',
            'IEEE738' : RCCIEEE,
            'CIGRE601' : RCCCIGRE,
            'ucIEEE738' : ucIEEE738,
            'ucCIGRE601' : ucCIGRE601}

        new_row_type = {
            'LineName' : str,
            'NodeName' : str,
            'TimeStamp' : str,
            'PhaseCurrent' : float, 
            'PhasePhase' : float, 
            'ConductorTemp' : float, 
            'AmbientTemp' : float, 
            'WindSpeed' : float, 
            'WindDomDirection' : float, 
            'WindAvgDirection' : float, 
            'SolarRadiation' : float, 
            'DewPoint' : float, 
            'ServiceName' : str,
            'Clearance' : Clearance,
            'IMAX' : float,
            'LoadMVA' : float, 
            'MaxCapacityMVA' :  float, 
            'IEEE738' : float, 
            'CIGRE601' : float, 
            'ucIEEE738' : float, 
            'ucCIGRE601' : float}


        CIGREerror = X2.get_error()
        if CIGREerror == 0:
            new_df = pd.DataFrame( [new_row_data]) #, dtype=new_row_data)
            df2 = pd.concat( [df2, new_df], ignore_index=True)
        else:
            TotalErrors += 1

        
        TotalCount += 1
        PrintControl += 1
        
        if PrintControl == PrintEach:
            PrintControl = 1
            if TotalRows > 0:
                PercTotal = round(100*TotalCount/TotalRows,1) 
                print('\r' + TimeStamp + '; ' + LineName + '; ' + NodeName + '; Percentage: ' + str(PercTotal) + ' %                                 ', end=' ', flush=True)    
                df2.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
            else:
                print('\r No data in this file...')
            df2 = pd.DataFrame( columns = column_names)

    
    df2.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
    if TotalRows > 0:
        PercTotal = round(100*TotalCount/TotalRows,2) 
        print('\r' + TimeStamp + '; ' + LineName + '; ' + NodeName + '; Percentage: ' + str(PercTotal) + ' %  ; Total Count [rows]: ' + str(TotalCount) + '                                    ', end=' ', flush=True)    
    else:
        print('\r No data in this file...')  
     
    return( TotalErrors)
    

In [10]:

dtype_mapping = {
            'LineName' : 'text',
            'NodeName' : 'text',
            'TimeStamp' : 'timestamp',
            'PhaseCurrent' : 'double precision',
            'PhasePhase' : 'double precision',
            'ConductorTemp' : 'double precision',
            'AmbientTemp' : 'double precision',
            'WindSpeed' : 'double precision',
            'WindDomDirection' : 'double precision',
            'WindAvgDirection' : 'double precision',
            'SolarRadiation' : 'double precision',
            'DewPoint' : 'double precision',
            'ServiceName' : 'text',
            'Clearance' : 'double precision',
            'IMAX' : 'double precision',
            'LoadMVA' : 'double precision',
            'MaxCapacityMVA' : 'double precision',
            'IEEE738' : 'double precision',
            'CIGRE601' : 'double precision',
            'ucIEEE738' : 'double precision',
            'ucCIGRE601' : 'double precision'}


Errors = 0 # Total number of errors and inconsistencies

for index_p, value_p in enumerate( patha):
    file_list = os.listdir( patha[index_p])
    print( file_list)
    PV1 = pvsystems.PVSystems()
    
    
    for index, value in enumerate( file_list):
        #LineName = LineNamea[ index_p]
        #split_list = value.split(" ")
        #NodeName = split_list[1]
        
        #print( NodeName)
        #YearMonth_list = split_list[2].split(".") 
        #YearMonth = YearMonth_list[0]
        #print( YearMonth)
        #y1 = YearMonth[0]
        #y2 = YearMonth[1]
        #yy = y1 + y2
        #year = 2000 + int( yy)
        #print(year)
        #m1 = YearMonth[2]
        #m2 = YearMonth[3]
        #mm = m1 + m2
        #month = int(mm)
        fullpath = patha[index_p] + file_list[index]
        print("%s " %(fullpath))
        inicio = time.time()
        #print( "%s ### %s - %s. %i / %i" %(fullpath, LineName, NodeName,  year, month))
        data_frame2 = pd.read_csv( fullpath, delimiter=';', header=0, decimal=',')

        #LineName = 'COLLADO - BUÑOL'
        #NodeName = '10006-10007'
        
        # Filter colums with missing values
        columns_to_check = [
            'TimeStamp', 
            'AmbientTemp', 
            'WindSpeed', 
            'WindDomDirection',
            'WindAvgDirection', 
            'SolarRadiation', 
            'PhaseCurrent', 
            'ConductorTemp' 
            ]
        data_frame3 = data_frame2.dropna( subset=columns_to_check)
        data_frame4 = data_frame3[ data_frame3['ServiceName'] == 'RCC CIGRE']
        data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')
        data_frame5 = data_frame4[ data_frame4['IMAX'] > 0.0]

        #LineName = 'El Palmar-Espinardo'
        #NodeName = '10160'

     
        ierrors = get_df2( data_frame5, dfCables, dfCases, table_name, engine)
        Errors += ierrors
        #print(df22)
        #df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid', dtype=dtype_mapping) 
        
        ##df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
        fin = time.time()
        tiempo_total = round((fin - inicio)/60.0,1)
        print("                                                                   ")
        print(f'Execution time: {tiempo_total} minutos')
        print('Inconsistencies: %i' %(ierrors))
        print("********************************************************************************")
        print("********************************************************************************")


print("Total number of inconsistencies: %i" %(Errors))             
        
    

['GE_20210101_20210131.csv', 'GE_20210201_20210228.csv', 'GE_20210301_20210331.csv', 'GE_20210401_20210430.csv', 'GE_20210501_20210531.csv', 'GE_20210601_20210630.csv', 'GE_20210701_20210731.csv', 'GE_20210801_20210831.csv', 'GE_20210901_20210930.csv', 'GE_20211001_20211031.csv', 'GE_20211101_20211130.csv', 'GE_20211201_20211231.csv', 'GE_20220101_20220131.csv', 'GE_20220201_20220228.csv', 'GE_20220301_20220331.csv', 'GE_20220401_20220430.csv', 'GE_20220501_20220531.csv', 'GE_20220601_20220630.csv', 'GE_20220701_20220731.csv', 'GE_20220801_20220831.csv', 'GE_20220901_20220930.csv', 'GE_20221001_20221031.csv', 'GE_20221101_20221130.csv', 'GE_20221201_20221231.csv', 'GE_20230101_20230131.csv', 'GE_20230201_20230228.csv', 'GE_20230301_20230331.csv', 'GE_20230401_20230430.csv', 'GE_20230501_20230531.csv', 'GE_20230601_20230630.csv', 'GE_20230701_20230731.csv', 'GE_20230801_20230831.csv', 'GE_20230901_20230930.csv', 'GE_20231001_20231031.csv', 'GE_20231101_20231130.csv']
E:/GE_Iberdrola/GE_

C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:55: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame2 = pd.read_csv( fullpath, delimiter=';', header=0, decimal=',')
C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')


2022-12-31 23:59:00; ROCAMORA - CARRUS; 19001-19002; Percentage: 100.0 %  ; Total Count [rows]: 114748                                                                                                        
Execution time: 5.3 minutos
Inconsistencies: 0
********************************************************************************
********************************************************************************
E:/GE_Iberdrola/GE_20230101_20230131.csv 


C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:55: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame2 = pd.read_csv( fullpath, delimiter=';', header=0, decimal=',')
C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')


2023-01-31 23:59:00; ROCAMORA - CARRUS; 19001-19002; Percentage: 100.0 %  ; Total Count [rows]: 309176                                                                                                        
Execution time: 14.1 minutos
Inconsistencies: 0
********************************************************************************
********************************************************************************
E:/GE_Iberdrola/GE_20230201_20230228.csv 


...

2023-09-30 23:19:00; ROCAMORA - CARRUS; 19001-19002; Percentage: 100.0 %  ; Total Count [rows]: 530448                                                                                                        
Execution time: 23.5 minutos
Inconsistencies: 0
********************************************************************************
********************************************************************************
E:/GE_Iberdrola/GE_20231001_20231031.csv 


C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')


2023-10-13 10:25:00; ROCAMORA - CARRUS; 19001-19002; Percentage: 100.0 %  ; Total Count [rows]: 236147                                                                                                        
Execution time: 10.2 minutos
Inconsistencies: 0
********************************************************************************
********************************************************************************
E:/GE_Iberdrola/GE_20231101_20231130.csv 


C:\Users\manan\AppData\Local\Temp\ipykernel_10688\3701740023.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')


2023-11-30 18:12:00; ROCAMORA - CARRUS; 19001-19002; Percentage: 100.0 %  ; Total Count [rows]: 323251                                                                                                        
Execution time: 14.6 minutos
Inconsistencies: 0
********************************************************************************
********************************************************************************
Total number of inconsistencies: 0


In [11]:
# Close the database connection
engine.dispose()